# Assumption Ops: supplier disruption walkthrough

Run **Runtime → Run all** in Google Colab. This CPU-only example needs no API keys or external business accounts. The bootstrap installs the repository only if `assumption_ops` is unavailable.

We allocate stock to two orders, accept a conflicting supplier update, inspect affected decisions, resolve the evidence conflict, and compute a replacement plan. Approval and execution are separate: execution writes a **local intent record**, with no ERP or supplier side effects.

The core is structured evidence plus exact constraints. A supplier message becomes a proposed field update; a person explicitly accepts and resolves it. No language model silently changes operational facts.


For CSV intake, compact business metrics, and ranked responses to the delay, run [the operational comparison notebook](https://colab.research.google.com/github/crypticsymmetry/improved-giggle/blob/main/notebooks/operational_scenarios.ipynb). The delay simulation and confirmed replan now use the same prior allocation, so their costs are directly comparable.

In [ ]:
from collections import defaultdict
import importlib.util
import subprocess
import sys

needs_install = importlib.util.find_spec("assumption_ops") is None
if not needs_install:
    import assumption_ops as installed_ops
    needs_install = not all(hasattr(installed_ops, name) for name in ("ScenarioComparison", "EvidenceUpdate"))
if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        "git+https://github.com/crypticsymmetry/improved-giggle.git",
    ])
    # Refresh only this framework if an older version was imported in this runtime.
    for module_name in list(sys.modules):
        if module_name == "assumption_ops" or module_name.startswith("assumption_ops."):
            del sys.modules[module_name]

from assumption_ops import (
    ATMS, OperationsPipeline, Supply, Order, Policy, Scenario,
    StaleDecisionError, evaluate_plan, load_csv_scenario, write_comparison_csv,
)
from assumption_ops.evidence import EvidenceConflict
from pprint import pprint


## Minimal alternative assumptions: a separate ATMS example

`ready` can follow from stock, shipment, or a spare together with substitution approval. The reasoner retains all three minimal support environments when the current context disables one of them. This preserves explanations across hypothetical contexts.

These logical alternatives alone cannot enforce shared inventory quantities. The allocation optimizer handles quantity, deadline, and substitution constraints in the operational example below.

In [ ]:
reasoner = ATMS(max_environments=100)
for assumption in ("stock", "shipment", "spare", "substitution_policy"):
    reasoner.add_assumption(assumption)
reasoner.add_rule("ready", ("stock",))
reasoner.add_rule("ready", ("shipment",))
reasoner.add_rule("ready", ("spare", "substitution_policy"))
expected_supports = {
    frozenset({"stock"}),
    frozenset({"shipment"}),
    frozenset({"spare", "substitution_policy"}),
}
assert set(reasoner.supports("ready")) == expected_supports
pprint(reasoner.explain("ready"))

reasoner.set_active("shipment", False)
assert reasoner.is_supported("ready")
assert set(reasoner.supports("ready")) == expected_supports
print("Shipment disabled: stock or an approved spare still supports ready.")
pprint(reasoner.explain("ready"))

reasoner.set_active("stock", False)
assert reasoner.is_supported("ready")
assert set(reasoner.supports("ready")) == expected_supports
assert reasoner.explain("ready")["active_supports"] == [["spare", "substitution_policy"]]
print("Stock also disabled: only the approved spare supports ready.")
pprint(reasoner.explain("ready"))

## 1. Seed an explicit operational world

Day numbers are relative to the start of this scenario. A shipment due on day 2 can fulfill an order due on day 3. SKU B is an approved substitute for A.

The planner minimizes its configured weighted cost, including priority-weighted unfilled penalties. Priority is a weight, not a guarantee of lexicographic service order. Availability and quantity remain hard constraints.


In [ ]:
supplies = [
    Supply("stock", "A", 3, 0, 1),
    Supply("shipment", "A", 7, 2, 1),
    Supply("substitute", "B", 4, 0, 3),
]
orders = [
    Order("urgent", "A", 5, 3, 3),
    Order("standard", "A", 5, 4, 1),
]
policy = Policy(substitutions={"A": ("B",)})

pipeline = OperationsPipeline(db_path=":memory:")
pipeline.seed(supplies=supplies, orders=orders, policy=policy)
baseline = pipeline.plan()
pprint(baseline.plan.to_dict())


## 2. Check the actual allocation constraints

These checks inspect the returned allocations rather than merely trusting the displayed explanation. We reuse them after the shipment delay.


In [ ]:
def validate_allocations(result, supply_rows, order_rows, substitution_policy):
    supply_by_id = {s.id: s for s in supply_rows}
    order_by_id = {o.id: o for o in order_rows}
    supply_usage = defaultdict(int)
    order_fill = defaultdict(int)
    for allocation in result.plan.allocations:
        supply = supply_by_id[allocation.supply_id]
        order = order_by_id[allocation.order_id]
        assert allocation.quantity > 0
        assert supply.available_day <= order.due_day, "Late supply was allocated"
        allowed = {order.sku, *substitution_policy.substitutions.get(order.sku, ())}
        assert supply.sku in allowed, "Unapproved substitution was allocated"
        supply_usage[supply.id] += allocation.quantity
        order_fill[order.id] += allocation.quantity
    assert all(supply_usage[s.id] <= s.quantity for s in supply_rows)
    assert all(order_fill[o.id] <= o.quantity for o in order_rows)
    for order in order_rows:
        assert order_fill[order.id] + result.plan.unfilled.get(order.id, 0) == order.quantity
    return {"supply_usage": dict(supply_usage), "order_fill": dict(order_fill)}

pprint(validate_allocations(baseline, supplies, orders, policy))
# Retain one approved decision to demonstrate the freshness gate later.
old_decision = next(d for d in baseline.decisions if d.order_id == "urgent")
pipeline.approve(old_decision.id)


## Read-only what-if: test a delay before accepting evidence

The hypothetical planner replaces only the arrival date in a temporary snapshot. It evaluates the disruption relative to the baseline plan without creating facts, decisions, or audit events. The assertions verify this read-only contract.

In [ ]:
revision_before = pipeline.store.revision
evidence_events_before = pipeline.store.events()
operations_events_before = pipeline.events()

delay_scenario = pipeline.what_if(
    {"supply:shipment": {"available_day": 8}}, previous=baseline.plan
)
pprint({
    "baseline": baseline.plan.to_dict(),
    "hypothetical_delay": delay_scenario.to_dict(),
})
assert pipeline.store.revision == revision_before
assert pipeline.store.events() == evidence_events_before
assert pipeline.events() == operations_events_before
assert sum(baseline.plan.unfilled.values()) == 0
assert sum(delay_scenario.unfilled.values()) >= 3
assert all(a.supply_id != "shipment" for a in delay_scenario.allocations)
print("What-if left accepted evidence and both audit histories unchanged.")

## 3. Accept a supplier update and inspect the conflict

The original evidence says day 2; the new supplier email says day 8. Acceptance admits the new evidence. Resolution explicitly chooses which accepted value becomes operational. Until then, the system must expose the disagreement rather than silently overwrite the old fact.


In [ ]:
update_id = pipeline.propose(
    "supply:shipment", "available_day", 8,
    "Supplier email: shipment delayed; expected arrival is now day 8.",
)
pipeline.accept(update_id)
pprint(pipeline.impact())

try:
    pipeline.plan()
except EvidenceConflict as exc:
    print(f"Expected unresolved-evidence rejection: {exc}")
else:
    raise AssertionError("Planning proceeded while accepted evidence conflicted")


## 4. Resolve the disputed date and recompute

The shipment cannot now fulfill either order on time. Remaining stock and approved substitutes can fill at most seven of the ten requested units. The new plan must report the remaining demand rather than invent capacity or use late stock.


In [ ]:
pipeline.resolve("supply:shipment", "available_day", update_id)
revised = pipeline.plan(previous=baseline.plan)
pprint(revised.plan.to_dict())

revised_supplies = [
    Supply("stock", "A", 3, 0, 1),
    Supply("shipment", "A", 7, 8, 1),
    Supply("substitute", "B", 4, 0, 3),
]
pprint(validate_allocations(revised, revised_supplies, orders, policy))
assert sum(a.quantity for a in revised.plan.allocations) <= 7
assert sum(revised.plan.unfilled.values()) >= 3
assert all(a.supply_id != "shipment" for a in revised.plan.allocations)

assert revised.plan.objective == delay_scenario.objective
print("Matched what-if and confirmed replan use the same previous-plan penalty:")
pprint(evaluate_plan(revised_supplies, orders, policy, revised.plan, baseline.plan).to_dict()["costs"])


## 5. Prove that yesterday's approval cannot execute today

The approved baseline decision depends on an obsolete evidence snapshot. Execution must fail after the update, even though a person approved that decision earlier.


In [ ]:
try:
    pipeline.commit(old_decision.id)
except StaleDecisionError as exc:
    print(f"Expected freshness rejection: {exc}")
else:
    raise AssertionError("An outdated approved decision was allowed to execute")


## 6. Inspect why the new decision is justified

The explanation exposes structured support evidence, allocation, and decision status. It is intended to be inspectable by applications and people; it does not depend on generated narrative being correct.


In [ ]:
new_decision = next(d for d in revised.decisions if d.order_id == "urgent")
pprint(pipeline.explain(new_decision.id))


## 7. Approve and record a local execution intent

This demonstration records intent only. A production ERP adapter must add authorization, idempotency, reconciliation, and its own transactional controls before creating a real order or changing a customer promise.


In [ ]:
pipeline.approve(new_decision.id)
execution = pipeline.commit(new_decision.id)
pprint(execution)


### Idempotency and inventory reservations

Repeating the same commit returns the existing result without recording another intent. A new plan uses only the inventory and demand remaining after the committed allocation. Reservations protect local intent from double allocation; they do not confirm that an external ERP executed it.

In [ ]:
events_after_commit = pipeline.events()
repeat_execution = pipeline.commit(new_decision.id)
assert repeat_execution["idempotent"] is True
assert pipeline.events() == events_after_commit
pprint(repeat_execution)

reserved_supply = defaultdict(int)
reserved_order = defaultdict(int)
for allocation in new_decision.allocations:
    reserved_supply[allocation.supply_id] += allocation.quantity
    reserved_order[allocation.order_id] += allocation.quantity
remaining_supplies = [
    Supply(s.id, s.sku, s.quantity - reserved_supply[s.id], s.available_day, s.unit_cost)
    for s in revised_supplies
]
remaining_orders = [
    Order(o.id, o.sku, o.quantity - reserved_order[o.id], o.due_day, o.priority)
    for o in orders
]
post_commit = pipeline.plan(previous=revised.plan)
pprint(post_commit.to_dict())
pprint(v…68 tokens truncated…units already reserved by the committed decision.")

## 8. Try your own scenario

Edit this ordinary JSON-compatible dictionary and rerun the cell. `examples/sample_scenario.json` contains the same starting scenario in the repository. This cell creates an independent database so changes do not mix with the earlier evidence history.

Try reducing substitute capacity, forbidding substitutions, changing order priority, or moving the shipment past the due dates. Inspect unfilled demand and the exact allocations after every change.


In [ ]:
import json

scenario = {
    "supplies": [
        {
            "id": "stock",
            "sku": "A",
            "quantity": 3,
            "available_day": 0,
            "unit_cost": 1
        },
        {
            "id": "shipment",
            "sku": "A",
            "quantity": 7,
            "available_day": 2,
            "unit_cost": 1
        },
        {
            "id": "substitute",
            "sku": "B",
            "quantity": 4,
            "available_day": 0,
            "unit_cost": 3
        }
    ],
    "orders": [
        {
            "id": "urgent",
            "sku": "A",
            "quantity": 5,
            "due_day": 3,
            "priority": 3
        },
        {
            "id": "standard",
            "sku": "A",
            "quantity": 5,
            "due_day": 4,
            "priority": 1
        }
    ],
    "policy": {
        "substitutions": {
            "A": [
                "B"
            ]
        }
    }
}

# Optional: replace the dictionary with an uploaded JSON file in Colab.
# from pathlib import Path
# scenario = json.loads(Path("sample_scenario.json").read_text())
custom_supplies = [Supply(**row) for row in scenario["supplies"]]
custom_orders = [Order(**row) for row in scenario["orders"]]
custom_policy = Policy(
    substitutions={sku: tuple(options) for sku, options in scenario["policy"]["substitutions"].items()}
)
custom_pipeline = OperationsPipeline(db_path=":memory:")
custom_pipeline.seed(supplies=custom_supplies, orders=custom_orders, policy=custom_policy)
custom_result = custom_pipeline.plan()
pprint(custom_result.plan.to_dict())
pprint(validate_allocations(custom_result, custom_supplies, custom_orders, custom_policy))


## Next implementation experiments

- Add a new evidence update and measure which decision supports become invalid.
- Compare priority-weighted plans under several penalty settings.
- Replay real exception cases after replacing identifiers and removing sensitive data.
- Persist the database by choosing a file path; verify evidence and decision history across process restarts.

This repository is a bounded operational prototype. A general ATMS can maintain alternative minimal assumption environments; this walkthrough exercises the implemented decision-support provenance and fail-closed evidence lifecycle. Avoid treating the two capabilities as equivalent.
